
<div style="text-align: center; padding: 30px 10px;">

<h1 style="color:#ff7500; font-size: 24px; margin-bottom: 10px;">
МФТИ ФПМИ
</h1>

<h2 style="font-size: 30px; margin-top: 5px;">
Практикум Python - Продвинутый Поток
</h2>

<hr style="width: 60%; border: 1px solid #10069f; margin: 25px auto;">

<h3 style="font-size: 36px;">
11. Typing. Type hints. Практика.
</h3>

<p style="margin-top: 20px;">
<strong>Дата:</strong> 14 апреля - 16 апреля 2026 года<br>
</p>

<p style="margin-top: 25px;">
Данный ноутбук является частью серии семинаров по курсу  
<em>«Практикум Python»</em> и предназначен для учебных и образовательных целей.
</p>

</div>

In [44]:
from IPython.core.magic import register_cell_magic
import tempfile
import os
import sys
from mypy import api

STRICT_FLAGS = [
    "--show-error-codes",
    "--no-error-summary",
    "--pretty",
]

@register_cell_magic
def mypy(line, cell):
    """
    Strict mypy typechecking cell magic.

    Usage:
        %%mypy
        x: tuple[float] = (1.0, 2.0)
    """

    # Write cell to temporary module
    with tempfile.TemporaryDirectory() as tmpdir:
        module_path = os.path.join(tmpdir, "mypy_cell.py")

        with open(module_path, "w", encoding="utf-8") as f:
            f.write(cell)

        # Run mypy
        result = api.run([
            module_path,
            *STRICT_FLAGS,
        ])

        stdout, stderr, exit_code = result

        if stdout:
            print(stdout)

        if stderr:
            print(stderr, file=sys.stderr)

        if exit_code == 0:
            print("✅ mypy strict typecheck passed")

In [5]:
!pip install mypy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.3/230.3 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 2.0 MB/s eta 0:00:00


In [8]:
import typing

## Вступление. Как работает проверка типов?

- типы проверяются не по реальному рантайм типу объекта, а по значению тайпхинта
- когда происходит присвоение переменной/биндинг аргумента функции, mypy проверяет, что передан подтип типа переменной/параметра
- если тип переменной не указан, тип переменной выводится по самому узкому возможному типу
- когда объявляется класс-наследник, mypy проверяет что наследник, действительно, является подтипом родителя

Определение подтипа:
* ∀ A: A -> A
* A -> B  =>  A.values ⊇ B.values
* A -> B  =>  A.functions ⊆ B.functions

\* логика такова, что в переменную a типа А можно положить объект любого его подтипа, и код, предполагающий что `a` имеет тип `A`, не сломается

<b><font color="#FF69B4"> Вопрос: </font></b> Как регулировать логику подклассов в рантайме, динамически?

<b><font color="#FF69B4"> Ваш ответ здесь </font></b>



<b><font color="#FF69B4"> Вопрос: </font></b> Использует ли mypy (статический линтер) `__subclasshook__`?

<b><font color="#FF69B4"> Ваш ответ здесь </font></b>

<b><font color="#FF69B4"> Вопрос: </font></b> Как тогда настраивать логику подклассов без наследования в mypy/статических линтерах?

<b><font color="#FF69B4"> Ваш ответ здесь </font></b>

## А если тип не указан?

Если тип чего-либо не указан, то этому объекту автоматически присваивается специальный тип Any. Any имеет свойства, которые можно интерпретировать как то, что Any является подтипом любого типа

* A -> B => A ~> B
* ∀ A: Any ~> A
* ∀ A: A ~> Any  

In [ ]:
%%mypy
# Задаем все типы
def f(a: int) -> None:
    reveal_type(a)
    print(a << 10)

def g() -> None:
    b = 1.0
    reveal_type(b)
    f(b)

<string>:4: note: Revealed type is "int"
<string>:9: note: Revealed type is "float"
<string>:10: error: Argument 1 to "f" has incompatible type "float"; expected "int"  [arg-type]
Found 1 error in 1 file (checked 1 source file)



In [ ]:
%%mypy
# Не задаем типы в f
def f(a):
    c: int = a
    reveal_type(a)
    reveal_type(c)
    print(a << 10)

def g() -> None:
    b = 1.0
    reveal_type(b)
    f(b)

<string>:4: note: By default the bodies of untyped functions are not checked, consider using --check-untyped-defs  [annotation-unchecked]
<string>:5: note: Revealed type is "Any"
<string>:5: note: 'reveal_type' always outputs 'Any' in unchecked functions
<string>:6: note: Revealed type is "Any"
<string>:6: note: 'reveal_type' always outputs 'Any' in unchecked functions
<string>:11: note: Revealed type is "float"
Success: no issues found in 1 source file



In [ ]:
%%mypy
# Не задаем типы в g
def f(a: int) -> None:
    reveal_type(a)
    print(a << 10)

def g():
    b = 1.0
    reveal_type(b)
    f(b)

<string>:4: note: Revealed type is "int"
<string>:9: note: Revealed type is "Any"
<string>:9: note: 'reveal_type' always outputs 'Any' in unchecked functions
Success: no issues found in 1 source file



In [ ]:
%%mypy
# Задаем типы для f частично (без a)
def f(a) -> None:
    reveal_type(a)
    print(a << 10)

def g() -> None:
    b = 1.0
    reveal_type(b)
    f(b)

<string>:4: note: Revealed type is "Any"
<string>:9: note: Revealed type is "float"
Success: no issues found in 1 source file



## Задание 1. Начало.

Аннотируйте функцию.

In [ ]:
%%mypy
# Задача
def f(a):
    return a / 2


# тесты
def tests():
    f("dd") # ERROR
    f(1j) # ERROR
    f(1) # SUCCESS
    f(1.0) # SUCCESS
    class R(float): # SUCCESS
        pass

    f(R(1))

Success: no issues found in 1 source file



<b><font color="#FF69B4"> Вопрос: </font></b> Почему такой вариант НЕ подходит?

```
from typing import Protocol

class SupportsDiv(Protocol):
    def __truediv__(self, other: int) -> float: ...

def f(a: SupportsDiv) -> float:
    return a / 2
```


## Задание 1.5

Реши задание 1. без union/any.

In [ ]:
%%mypy
# Задача
def f(a):
    return a / 2


# тесты
def tests():
    f("dd") # ERROR
    f(1j) # ERROR
    f(1) # SUCCESS
    f(1.0) # SUCCESS
    class R(float): # SUCCESS
        pass

    f(R(1))

# Задание 2. Гомогенный tuple.

Аннотируйте функцию.



In [40]:
%%mypy
def f(a: tuple[float, ...]) -> float | None:
    return a[0] if a else None


def tests() -> None:
    f((1, 1, 2, 4)) # SUCCESS
    f((1,)) # SUCCESS
    f((1.2, 3.4)) # SUCCESS
    f((1j, 3j, 6j, 7j)) # ERROR
    f((True, False)) # SUCCESS
    f(("there", "are", "no", "reason")) # ERROR
    f(("there", 1)) # ERROR


/tmp/tmpcx1r_ob5/mypy_cell.py:9: error: Argument 1 to "f" has incompatible type
"tuple[complex, complex, complex, complex]"; expected "tuple[float, ...]" 
[arg-type]
        f((1j, 3j, 6j, 7j)) # ERROR
          ^~~~~~~~~~~~~~~~
/tmp/tmpcx1r_ob5/mypy_cell.py:11: error: Argument 1 to "f" has incompatible
type "tuple[str, str, str, str]"; expected "tuple[float, ...]"  [arg-type]
        f(("there", "are", "no", "reason")) # ERROR
          ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
/tmp/tmpcx1r_ob5/mypy_cell.py:12: error: Argument 1 to "f" has incompatible
type "tuple[str, int]"; expected "tuple[float, ...]"  [arg-type]
        f(("there", 1)) # ERROR
          ^~~~~~~~~~~~



# Задание 3. Гомогенный список.

<b><font color="#FF69B4"> Вопрос: </font></b> В чем различаются примеры ниже? Почему?

In [42]:
%%mypy
def f(a: tuple[float, ...]) -> float | None:
    return a[0] if a else None


def tests() -> None:
    f((1, 1, 2, 4)) # SUCCESS
    f((1,)) # SUCCESS
    f((1.2, 3.4)) # SUCCESS
    f((1, 2, 3.4)) # SUCCESS
    f((1j, 3j, 6j, 7j)) # ERROR
    f((True, False)) # SUCCESS
    f(("there", "are", "no", "reason")) # ERROR
    f(("there", 1)) # ERROR

/tmp/tmpae68b4un/mypy_cell.py:10: error: Argument 1 to "f" has incompatible
type "tuple[complex, complex, complex, complex]"; expected "tuple[float, ...]" 
[arg-type]
        f((1j, 3j, 6j, 7j)) # ERROR
          ^~~~~~~~~~~~~~~~
/tmp/tmpae68b4un/mypy_cell.py:12: error: Argument 1 to "f" has incompatible
type "tuple[str, str, str, str]"; expected "tuple[float, ...]"  [arg-type]
        f(("there", "are", "no", "reason")) # ERROR
          ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
/tmp/tmpae68b4un/mypy_cell.py:13: error: Argument 1 to "f" has incompatible
type "tuple[str, int]"; expected "tuple[float, ...]"  [arg-type]
        f(("there", 1)) # ERROR
          ^~~~~~~~~~~~



In [45]:
%%mypy
def f(a: list[float]) -> float | None:
    return a[0] if a else None


def tests() -> None:
    f([1, 1, 2, 4])
    f([1])
    f([1.2, 3.4])
    f([1, 2, 3.4])
    f([1j, 3j, 6j, 7j])
    f([True, False])
    f(["there", "are", "no", "reason"])
    f(["there", 1])

/tmp/tmpr92y7wvs/mypy_cell.py:10: error: List item 0 has incompatible type
"complex"; expected "float"  [list-item]
        f([1j, 3j, 6j, 7j])
           ^~
/tmp/tmpr92y7wvs/mypy_cell.py:10: error: List item 1 has incompatible type
"complex"; expected "float"  [list-item]
        f([1j, 3j, 6j, 7j])
               ^~
/tmp/tmpr92y7wvs/mypy_cell.py:10: error: List item 2 has incompatible type
"complex"; expected "float"  [list-item]
        f([1j, 3j, 6j, 7j])
                   ^~
/tmp/tmpr92y7wvs/mypy_cell.py:10: error: List item 3 has incompatible type
"complex"; expected "float"  [list-item]
        f([1j, 3j, 6j, 7j])
                       ^~
/tmp/tmpr92y7wvs/mypy_cell.py:12: error: List item 0 has incompatible type
"str"; expected "float"  [list-item]
        f(["there", "are", "no", "reason"])
           ^~~~~~~
/tmp/tmpr92y7wvs/mypy_cell.py:12: error: List item 1 has incompatible type
"str"; expected "float"  [list-item]
        f(["there", "are", "no", "reason"])
             

<b><font color="#FF69B4"> Вопрос: </font></b> В чем различаются примеры ниже? Почему?

In [47]:
%%mypy
class A:
    pass

class B(A):
    pass

class C(B):
    pass

def f(a: tuple[B, ...]) -> B | None:
    return a[0] if a else None


def tests() -> None:
    f((A(), A()))
    f((A(), B()))
    f((B(), B()))
    f((B(), C()))
    f((C(), C()))

/tmp/tmp4ipsck2f/mypy_cell.py:15: error: Argument 1 to "f" has incompatible
type "tuple[A, A]"; expected "tuple[B, ...]"  [arg-type]
        f((A(), A()))
          ^~~~~~~~~~
/tmp/tmp4ipsck2f/mypy_cell.py:16: error: Argument 1 to "f" has incompatible
type "tuple[A, B]"; expected "tuple[B, ...]"  [arg-type]
        f((A(), B()))
          ^~~~~~~~~~



In [49]:
%%mypy
class A:
    pass

class B(A):
    pass

class C(B):
    pass

def f(a: list[B]) -> B | None:
    return a[0] if a else None


def tests() -> None:
    f([A(), A()])
    f([A(), B()])
    f([B(), B()])
    f([B(), C()])
    f([C(), C()])

/tmp/tmp0rs7lsy3/mypy_cell.py:15: error: List item 0 has incompatible type "A";
expected "B"  [list-item]
        f([A(), A()])
           ^~~
/tmp/tmp0rs7lsy3/mypy_cell.py:15: error: List item 1 has incompatible type "A";
expected "B"  [list-item]
        f([A(), A()])
                ^~~
/tmp/tmp0rs7lsy3/mypy_cell.py:16: error: List item 0 has incompatible type "A";
expected "B"  [list-item]
        f([A(), B()])
           ^~~



In [50]:
%%mypy
class A:
    pass

class B(A):
    pass

class C(B):
    pass

def f(a: list[B]) -> B | None:
    return a[0] if a else None


def tests() -> None:
    a1 = [A(), A()]
    a2 = [A(), B()]
    a3 = [B(), B()]
    a4 = [B(), C()]
    a5 = [C(), C()]
    f(a1)
    f(a2)
    f(a3)
    f(a4)
    f(a5)

/tmp/tmpqz810cot/mypy_cell.py:20: error: Argument 1 to "f" has incompatible
type "list[A]"; expected "list[B]"  [arg-type]
        f(a1)
          ^~
/tmp/tmpqz810cot/mypy_cell.py:21: error: Argument 1 to "f" has incompatible
type "list[A]"; expected "list[B]"  [arg-type]
        f(a2)
          ^~
/tmp/tmpqz810cot/mypy_cell.py:24: error: Argument 1 to "f" has incompatible
type "list[C]"; expected "list[B]"  [arg-type]
        f(a5)
          ^~
/tmp/tmpqz810cot/mypy_cell.py:24: note: "list" is invariant -- see https://mypy.readthedocs.io/en/stable/common_issues.html#variance
/tmp/tmpqz810cot/mypy_cell.py:24: note: Consider using "Sequence" instead, which is covariant



In [51]:
%%mypy
class A:
    pass

class B(A):
    pass

class C(B):
    pass

def f(a: tuple[B, ...]) -> B | None:
    return a[0] if a else None


def tests() -> None:
    a1 = (A(), A())
    a2 = (A(), B())
    a3 = (B(), B())
    a4 = (B(), C())
    a5 = (C(), C())
    f(a1)
    f(a2)
    f(a3)
    f(a4)
    f(a5)

/tmp/tmpe32dvs_d/mypy_cell.py:20: error: Argument 1 to "f" has incompatible
type "tuple[A, A]"; expected "tuple[B, ...]"  [arg-type]
        f(a1)
          ^~
/tmp/tmpe32dvs_d/mypy_cell.py:21: error: Argument 1 to "f" has incompatible
type "tuple[A, B]"; expected "tuple[B, ...]"  [arg-type]
        f(a2)
          ^~



## Задание 4. Странный tuple.

Придумай аннотацию, которая подойдёт во всех этих случаях.

In [52]:
%%mypy
def f(a):
    return a[1]


def tests() -> None:
    ex1 = ("a", "b", "c")
    ex2 = ("a", "b", ["c", "d", "e"])
    ex3 = ("a", "b", "c", "d")

    f(ex1) # SUCCESS
    f(ex2) # SUCCESS
    f(ex3) # ERROR

    class A(str):
        pass

    ex4 = ("a", A(), A()) # SUCCESS
    a: A = f(ex4)

    ex5 = (["a", "b"], "c", "d")
    f(ex5) # SUCCESS


    class B:
        def __len__(self) -> int:
            return 1
    ex6 = (B(), "c", "d")
    f(ex6) # SUCCESS


    class C:
        def __len__(self) -> int:
            return 1
    ex7 = (C(), "c", {"d", "e"})
    f(ex7) # ERROR



/tmp/tmppap9obpf/mypy_cell.py:18: note: By default the bodies of untyped functions are not checked, consider using --check-untyped-defs  [annotation-unchecked]

✅ mypy strict typecheck passed


## Напиши протокол


In [ ]:
%%mypy
class Gettable:
    pass


def get(container, index):
    if container:
        return container[index]

    return None


def case1() -> None:
    class A:
        def __init__(self, a: str):
            self._a = a

        def __getitem__(self, item: int) -> str:
            return self._a[item]

        def __len__(self) -> int:
            return len(self._a)

    get(A("hello"), 4)


def case2() -> None:
    class A:
        def __init__(self, a: str):
            self._a = a

        def __getitem__(self, item: int) -> str:
            return self._a[item]

    get(A("hello"), 4)


def case3() -> None:
    class A:
        def __getitem__(self, item: int) -> bool:
            return True

        def __len__(self) -> int:
            return 0

    get(A(), 4)


## Пары.

Напиши корректные тайп хинты через шаблоны.

In [ ]:
%%mypy
class Pair:
    def __init__(self, a, b): pass

    def sum(self): pass

    def first(self): pass

    def second(self): pass

    def __iadd__(self, pair): pass


def case1() -> None:
    Pair[int](1, 2.0)  # fail


def case2() -> None:
    Pair[int](1.0, 2)  # fail


def case3() -> None:
    Pair[int](1, 2)  # success


def case4() -> None:
    Pair[float](1.0, 2.0)  # success


def case5() -> None:
    Pair[float](1, 2)  # success


def case6() -> None:
    Pair[str]("a", "b")  # fail

## Угадай - 1

Что выведет после запуска ячейки ниже?

In [ ]:
%%mypy
import typing as tp

class A:
    pass

class B(A):
    pass

def g(f: tp.Callable[[A, B]]) -> None:
    pass

def f1(a: A) -> A:
    pass

def f2(a: A) -> B:
    pass

def f3(a: B) -> A:
    pass

def f4(a: B) -> B:
    pass

g(f1)
g(f2)
g(f3)
g(f4)

## Угадай - 2

Что выведет после запуска ячейки ниже?

In [ ]:
%%mypy
import typing as tp

def foo(a: tp.Iterable[str]) -> bool:
    b = len(a)
    c = sum(1 for i in a)
    return b == c

foo(["a", "b"])
foo("ab")
foo({"a": 2})

class A:
    def __len__(self) -> int:
        return 1

foo(A())

class B:
    def __iter__(self) -> tp.Iterator[int]:
        return iter([])

foo(B())

class C:
    def __iter__(self) -> tp.Iterator[str]:
        return iter([])

foo(C())

## Угадай - 3

Что выведет после запуска ячейки ниже?

In [ ]:
%%mypy
import typing as tp

class Fooable(tp.Protocol, tp.Sized):
    def foo(self, a: int) -> None:
        pass

class A(Fooable):
    def __len__(self) -> int:
        return 10

class B:
    def foo(self, a: int) -> None:
        pass

    def __len__(self) -> int:
        return 10

class C:
    def foo(self, a: int) -> None:
        pass

def foo(a: int) -> None:
    pass

def f(c: Fooable) -> None:
    c.foo(1)
    len(c)
    "10" in c

f([ ])
f(A())
f(B())
f(C())
f(foo)

## Угадай - 4


Что выведет после запуска ячейки ниже?

In [ ]:
%%mypy

import typing as tp

T = tp.TypeVar("T", bound=tp.SupportsFloat, covariant=True)

class A(tp.Generic[T]):
    def __init__(self, a: T) -> None:
        self.a = tp.SupportsFloat = a

    def increment(self) -> float:
        self.a = float(self._a) + 1
        return self._a

A(1)
A(1.2)
A(True)
A("1.3")

class B:
    def __float__(self) -> float:
        return 1.1

A(B())

def g(a: A[int]) -> None:
    pass

g(A(1.4))
g(A(True))